# Data Profiling - Urban Flow Analytics Taxi Dataset

**Run by:** [Praveen Madawalage]
**Date:** [2026/09/10]

Purpose: profile all 12 monthly CSVs before cleaning - check schema consistency,`
dtypes, row counts, and date-range containment. This notebook's findings feed
directly into the cleaning decisions and the technical report's preprocessing section.

In [ ]:
import duckdb
import glob
import pandas as pd

files = sorted(glob.glob("../data/raw/*.csv"))  # adjust path if running from notebooks/
print(f"Found {len(files)} files")
for f in files:
    print(" -", f)

## 1. File Inventory & Schema Scan

Scanning each file's row count, column names, and dtypes using DuckDB
(lazy CSV scan - doesn't load full files into RAM).

In [ ]:
con = duckdb.connect()
profile_rows = []

for f in files:
    schema = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{f}')").fetchdf()
    row_count = con.execute(f"SELECT COUNT(*) AS n FROM read_csv_auto('{f}')").fetchone()[0]

    profile_rows.append({
        "file": f,
        "n_rows": row_count,
        "n_cols": len(schema),
        "columns": tuple(schema["column_name"].tolist()),
        "dtypes": tuple(schema["column_type"].tolist()),
    })
    print(f"{f}: {row_count:,} rows, {len(schema)} cols")

profile_df = pd.DataFrame(profile_rows)
profile_df  

In [ ]:
# Save raw profile for the record / report appendix
profile_df.to_csv("../data/interim/file_schema_profile.csv", index=False)

## 2. Schema Consistency Check

- If unique column-name sets == 1 → all files match, safe to concatenate directly.
- If > 1 → inspect diffs below before merging (may be expected, e.g. a fee column
  introduced partway through the year per the data dictionary).

In [ ]:
unique_schemas = profile_df["columns"].nunique()
unique_dtypes = profile_df["dtypes"].nunique()
print(f"Unique column-name sets across {len(files)} files: {unique_schemas}")
print(f"Unique dtype sets across {len(files)} files: {unique_dtypes}")

if unique_schemas > 1:
    print("\n⚠️ SCHEMA MISMATCH DETECTED. Columns differ between files:")
    baseline = set(profile_df["columns"].iloc[0])
    for i, row in profile_df.iterrows():
        diff = set(row["columns"]).symmetric_difference(baseline)
        if diff:
            print(f"  {row['file']}: differs by {diff}")
else:
    print("\n✅ All files share the same column set.")

if unique_dtypes > 1:
    print("\n⚠️ DTYPE MISMATCH DETECTED — inspect per-column below.")
else:
    print("✅ All files share the same dtypes.")

output: 
✅ All files share the same column set.
✅ All files share the same dtypes.

## 3. Date Range Containment Check

Each file should contain pickups almost entirely within its labeled month.
Rows falling well outside that range suggest mislabeled or corrupted data.

In [39]:
date_range_rows = []

for f in files:
    result = con.execute(f"""
        SELECT 
            MIN(pickup_timestamp) AS min_pickup, 
            MAX(pickup_timestamp) AS max_pickup,
            COUNT(*) AS total
        FROM read_csv_auto('{f}')
    """).fetchdf()
    row = result.to_dict("records")[0]
    row["file"] = f
    date_range_rows.append(row)
    print(f, row)

date_range_df = pd.DataFrame(date_range_rows)
date_range_df.to_csv("../data/interim/file_date_ranges.csv", index=False)
date_range_df

../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv {'min_pickup': Timestamp('2025-03-31 23:45:01'), 'max_pickup': Timestamp('2025-05-01 00:48:13'), 'total': 3970553, 'file': '../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv'}
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv {'min_pickup': Timestamp('2009-01-01 00:20:39'), 'max_pickup': Timestamp('2025-06-01 00:04:31'), 'total': 4591845, 'file': '../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv'}
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv {'min_pickup': Timestamp('2025-05-31 22:34:26'), 'max_pickup': Timestamp('2025-06-30 23:59:59'), 'total': 4322960, 'file': '../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv'}
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv {'min_pickup': Timestamp('2009-01-01 08:52:26'), 'max_pickup': Timestamp('2025-07-31 23:59:59'), 'total': 3898963, 'file': '../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv'}
../data/raw/Urban_Flow_Analytics

,min_pickup,max_pickup,total,file
0,2025-03-31 23:45:01,2025-05-01 00:48:13,3970553,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
1,2009-01-01 00:20:39,2025-06-01 00:04:31,4591845,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
2,2025-05-31 22:34:26,2025-06-30 23:59:59,4322960,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
3,2009-01-01 08:52:26,2025-07-31 23:59:59,3898963,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
4,2009-01-01 12:52:15,2025-09-01 00:00:29,3574091,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
5,2025-08-31 23:45:38,2025-10-01 00:00:11,4251015,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
6,2025-09-30 22:54:51,2025-11-01 00:32:12,4428699,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
7,2008-12-31 23:04:21,2025-11-30 23:59:59,4181444,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
8,2025-11-30 20:48:56,2025-12-31 23:59:59,4305006,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
9,2025-12-31 23:57:29,2026-02-01 00:45:01,3724889,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv


### 3a. Date range findings

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>min_pickup</th>
      <th>max_pickup</th>
      <th>total</th>
      <th>file</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>2025-03-31 23:45:01</td>
      <td>2025-05-01 00:48:13</td>
      <td>3970553</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2009-01-01 00:20:39</td>
      <td>2025-06-01 00:04:31</td>
      <td>4591845</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv</td>
    </tr>
    <tr>
      <th>2</th>
      <td>2025-05-31 22:34:26</td>
      <td>2025-06-30 23:59:59</td>
      <td>4322960</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv</td>
    </tr>
    <tr>
      <th>3</th>
      <td>2009-01-01 08:52:26</td>
      <td>2025-07-31 23:59:59</td>
      <td>3898963</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv</td>
    </tr>
    <tr>
      <th>4</th>
      <td>2009-01-01 12:52:15</td>
      <td>2025-09-01 00:00:29</td>
      <td>3574091</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv</td>
    </tr>
    <tr>
      <th>5</th>
      <td>2025-08-31 23:45:38</td>
      <td>2025-10-01 00:00:11</td>
      <td>4251015</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv</td>
    </tr>
    <tr>
      <th>6</th>
      <td>2025-09-30 22:54:51</td>
      <td>2025-11-01 00:32:12</td>
      <td>4428699</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv</td>
    </tr>
    <tr>
      <th>7</th>
      <td>2008-12-31 23:04:21</td>
      <td>2025-11-30 23:59:59</td>
      <td>4181444</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv</td>
    </tr>
    <tr>
      <th>8</th>
      <td>2025-11-30 20:48:56</td>
      <td>2025-12-31 23:59:59</td>
      <td>4305006</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv</td>
    </tr>
    <tr>
      <th>9</th>
      <td>2025-12-31 23:57:29</td>
      <td>2026-02-01 00:45:01</td>
      <td>3724889</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv</td>
    </tr>
    <tr>
      <th>10</th>
      <td>2026-01-31 23:31:23</td>
      <td>2026-03-01 00:51:48</td>
      <td>3399866</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv</td>
    </tr>
    <tr>
      <th>11</th>
      <td>2008-12-31 23:03:20</td>
      <td>2026-04-01 00:06:25</td>
      <td>3952451</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv</td>
    </tr>
  </tbody>
</table>
</div>

In [ ]:
pd.set_option("display.max_colwidth", None)
date_range_df[["file", "min_pickup", "max_pickup", "total"]]

In [ ]:
anomaly_summary = []

for f in files:
    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN pickup_timestamp < '2020-01-01' THEN 1 ELSE 0 END) AS rows_before_2020,
            SUM(CASE WHEN pickup_timestamp > '2026-12-31' THEN 1 ELSE 0 END) AS rows_after_2026
        FROM read_csv_auto('{f}')
    """).fetchdf()
    row = result.to_dict("records")[0]
    row["file"] = f
    anomaly_summary.append(row)

anomaly_df = pd.DataFrame(anomaly_summary)
anomaly_df["pct_before_2020"] = 100 * anomaly_df["rows_before_2020"] / anomaly_df["total_rows"]
anomaly_df

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>min_pickup</th>
      <th>max_pickup</th>
      <th>total</th>
      <th>file</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>2025-03-31 23:45:01</td>
      <td>2025-05-01 00:48:13</td>
      <td>3970553</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2009-01-01 00:20:39</td>
      <td>2025-06-01 00:04:31</td>
      <td>4591845</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv</td>
    </tr>
    <tr>
      <th>2</th>
      <td>2025-05-31 22:34:26</td>
      <td>2025-06-30 23:59:59</td>
      <td>4322960</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv</td>
    </tr>
    <tr>
      <th>3</th>
      <td>2009-01-01 08:52:26</td>
      <td>2025-07-31 23:59:59</td>
      <td>3898963</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv</td>
    </tr>
    <tr>
      <th>4</th>
      <td>2009-01-01 12:52:15</td>
      <td>2025-09-01 00:00:29</td>
      <td>3574091</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv</td>
    </tr>
    <tr>
      <th>5</th>
      <td>2025-08-31 23:45:38</td>
      <td>2025-10-01 00:00:11</td>
      <td>4251015</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv</td>
    </tr>
    <tr>
      <th>6</th>
      <td>2025-09-30 22:54:51</td>
      <td>2025-11-01 00:32:12</td>
      <td>4428699</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv</td>
    </tr>
    <tr>
      <th>7</th>
      <td>2008-12-31 23:04:21</td>
      <td>2025-11-30 23:59:59</td>
      <td>4181444</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv</td>
    </tr>
    <tr>
      <th>8</th>
      <td>2025-11-30 20:48:56</td>
      <td>2025-12-31 23:59:59</td>
      <td>4305006</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv</td>
    </tr>
    <tr>
      <th>9</th>
      <td>2025-12-31 23:57:29</td>
      <td>2026-02-01 00:45:01</td>
      <td>3724889</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv</td>
    </tr>
    <tr>
      <th>10</th>
      <td>2026-01-31 23:31:23</td>
      <td>2026-03-01 00:51:48</td>
      <td>3399866</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv</td>
    </tr>
    <tr>
      <th>11</th>
      <td>2008-12-31 23:03:20</td>
      <td>2026-04-01 00:06:25</td>
      <td>3952451</td>
      <td>../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv</td>
    </tr>
  </tbody>
</table>
</div>

### 3b. Inspecting the pre-2020 anomalous rows

Only 8 rows total exhibit pickup_timestamp before 2020 (likely epoch/clock-reset
errors from onboard meter systems). Before deciding to drop them, inspect the
full row to check whether other fields are also invalid, or just the timestamp.

In [41]:
anomalous_rows = []

for f in files:
    result = con.execute(f"""
        SELECT *
        FROM read_csv_auto('{f}')
        WHERE pickup_timestamp < '2020-01-01'
    """).fetchdf()
    if len(result) > 0:
        result["source_file"] = f
        anomalous_rows.append(result)

if anomalous_rows:
    anomalous_df = pd.concat(anomalous_rows, ignore_index=True)
    print(f"Total anomalous rows found: {len(anomalous_df)}")
else:
    anomalous_df = pd.DataFrame()
    print("No anomalous rows found.")

anomalous_df

Total anomalous rows found: 8


,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,base_fare,surcharge_misc,transit_tax,driver_tip_payment,toll_total,service_improvement_fee,charge_total,zone_congestion_fee,Airport_fee,congestion_relief_fee,source_file
0,2,2009-01-01 00:20:39,2009-01-01 00:20:49,5.0,4.47,1.0,N,230,261,1,24.0,1.0,0.5,2.98,0.00,1.0,32.73,2.5,0.0,0.75,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
1,2,2009-01-01 08:52:26,2009-01-01 10:00:26,1.0,11.95,1.0,N,138,163,1,64.6,0.0,0.5,16.65,13.88,1.0,99.88,2.5,0.0,0.75,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
2,2,2009-01-01 12:52:15,2009-01-01 13:12:15,1.0,2.13,1.0,N,43,230,1,18.4,0.0,0.5,4.63,0.00,1.0,27.78,2.5,0.0,0.75,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
3,2,2008-12-31 23:04:21,2008-12-31 23:32:25,1.0,18.12,2.0,N,132,238,2,70.0,0.0,0.5,0.00,6.94,1.0,80.94,2.5,0.0,0.00,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
4,2,2009-01-01 00:05:50,2009-01-01 00:10:56,1.0,1.26,1.0,N,239,50,2,8.6,1.0,0.5,0.00,0.00,1.0,14.35,2.5,0.0,0.75,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
5,2,2009-01-01 14:08:04,2009-01-01 14:45:04,1.0,11.56,1.0,N,138,142,2,50.6,0.0,0.5,0.00,13.88,1.0,68.48,2.5,0.0,0.00,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
6,2,2009-01-01 00:02:17,2009-01-01 00:07:36,1.0,1.20,1.0,N,48,48,2,7.2,1.0,0.5,0.00,0.00,1.0,12.95,2.5,0.0,0.75,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv
7,2,2008-12-31 23:03:20,2009-01-01 14:24:52,1.0,16.90,3.0,N,48,1,1,83.2,0.0,0.0,15.00,22.79,1.0,122.74,0.0,0.0,0.75,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


Total anomalous rows found: 8
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>pickup_timestamp</th>
      <th>dropoff_timestamp</th>
      <th>trip_duration_min</th>
      <th>distance_miles</th>
      <th>avg_speed_mph</th>
      <th>base_fare</th>
      <th>charge_total</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>2009-01-01 00:20:39</td>
      <td>2009-01-01 00:20:49</td>
      <td>0.166667</td>
      <td>4.47</td>
      <td>1609.200000</td>
      <td>24.0</td>
      <td>32.73</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2009-01-01 08:52:26</td>
      <td>2009-01-01 10:00:26</td>
      <td>68.000000</td>
      <td>11.95</td>
      <td>10.544118</td>
      <td>64.6</td>
      <td>99.88</td>
    </tr>
    <tr>
      <th>2</th>
      <td>2009-01-01 12:52:15</td>
      <td>2009-01-01 13:12:15</td>
      <td>20.000000</td>
      <td>2.13</td>
      <td>6.390000</td>
      <td>18.4</td>
      <td>27.78</td>
    </tr>
    <tr>
      <th>3</th>
      <td>2008-12-31 23:04:21</td>
      <td>2008-12-31 23:32:25</td>
      <td>28.066667</td>
      <td>18.12</td>
      <td>38.736342</td>
      <td>70.0</td>
      <td>80.94</td>
    </tr>
    <tr>
      <th>4</th>
      <td>2009-01-01 00:05:50</td>
      <td>2009-01-01 00:10:56</td>
      <td>5.100000</td>
      <td>1.26</td>
      <td>14.823529</td>
      <td>8.6</td>
      <td>14.35</td>
    </tr>
    <tr>
      <th>5</th>
      <td>2009-01-01 14:08:04</td>
      <td>2009-01-01 14:45:04</td>
      <td>37.000000</td>
      <td>11.56</td>
      <td>18.745946</td>
      <td>50.6</td>
      <td>68.48</td>
    </tr>
    <tr>
      <th>6</th>
      <td>2009-01-01 00:02:17</td>
      <td>2009-01-01 00:07:36</td>
      <td>5.316667</td>
      <td>1.20</td>
      <td>13.542320</td>
      <td>7.2</td>
      <td>12.95</td>
    </tr>
    <tr>
      <th>7</th>
      <td>2008-12-31 23:03:20</td>
      <td>2009-01-01 14:24:52</td>
      <td>921.533333</td>
      <td>16.90</td>
      <td>1.100340</td>
      <td>83.2</td>
      <td>122.74</td>
    </tr>
  </tbody>
</table>
</div>

In [ ]:
anomalous_df["trip_duration_min"] = (
    pd.to_datetime(anomalous_df["dropoff_timestamp"]) - pd.to_datetime(anomalous_df["pickup_timestamp"])
).dt.total_seconds() / 60

anomalous_df["avg_speed_mph"] = anomalous_df["distance_miles"] / (anomalous_df["trip_duration_min"] / 60)

anomalous_df[["pickup_timestamp", "dropoff_timestamp", "trip_duration_min", 
              "distance_miles", "avg_speed_mph", "base_fare", "charge_total"]]

## 4. Decisions Going Into Cleaning

1. **Schema/dtype consistency**: ✅ All files share the same column set. ✅ All files share the same dtypes.

2. **File integrity / date alignment**: All 12 files are correctly labeled and contain
   the expected month's data, with normal minor spillover of a few hours into
   adjacent months at file boundaries. No mislabeled or corrupted files found.

3. **Corrupted timestamp anomaly**: 8 rows (0.0000165% of ~48.6M total) had
   pickup_timestamp values from 2008–2009, inconsistent with the dataset's expected
   2025–2026 range. Of these, 2 rows also failed physical plausibility checks
   (speed >1,600 mph or duration >15 hours for the given distance). Decision:
   drop all 8 rows — negligible row count, no measurable impact on downstream
   models, and avoids inconsistent dates corrupting time-based features used in
   demand forecasting.

4. **Outstanding checks before cleaning notebook is finalized**: negative fares,
   zero-distance trips with nonzero fare, zero passenger counts, dropoff-before-pickup
   (dataset-wide, not just the 8 rows above), and unrealistic speed/duration at scale —
   not yet quantified across the full ~48.6M rows.

## 5. Dataset-Wide Anomaly Quantification

Checking the anomaly types called out in the challenge brief, across all 12 files
combined (~48.6M rows), using DuckDB's glob support to query all files as one
logical table without loading them fully into memory.

Anomalies checked:
- Negative fares / negative charges
- Zero distance with nonzero fare
- Zero passenger count
- Dropoff before pickup
- Unrealistic speed (distance / duration)

In [ ]:
# Register all 12 files as a single virtual table via glob pattern
all_files_glob = "../data/raw/*.csv"

con.execute(f"""
    CREATE OR REPLACE VIEW taxi_all AS
    SELECT * FROM read_csv_auto('{all_files_glob}', union_by_name=True)
""")

total_rows = con.execute("SELECT COUNT(*) FROM taxi_all").fetchone()[0]
print(f"Total rows across all files: {total_rows:,}")

In [ ]:
anomaly_checks = {
    "negative_base_fare": "base_fare < 0",
    "negative_charge_total": "charge_total < 0",
    "zero_distance_nonzero_fare": "distance_miles = 0 AND base_fare > 0",
    "zero_passengers": "rider_count = 0",
    "dropoff_before_pickup": "dropoff_timestamp < pickup_timestamp",
    "zero_duration": "dropoff_timestamp = pickup_timestamp",
}

results = []
for name, condition in anomaly_checks.items():
    count = con.execute(f"SELECT COUNT(*) FROM taxi_all WHERE {condition}").fetchone()[0]
    pct = 100 * count / total_rows
    results.append({"anomaly": name, "count": count, "pct_of_total": round(pct, 6)})
    print(f"{name}: {count:,} rows ({pct:.6f}%)")

anomaly_summary_df = pd.DataFrame(results)
anomaly_summary_df.to_csv("../data/interim/anomaly_summary.csv", index=False)
anomaly_summary_df

### Unrealistic speed check

Speed = distance_miles / trip_duration_hours. Requires duration > 0 first
(zero/negative-duration rows are already flagged above and should be excluded
from this calculation to avoid divide-by-zero).

Threshold: NYC-legal speed limits and highway travel top out well under 80 mph
even on highways; we'll flag anything above 100 mph as implausible, and anything
with a very long duration + very short distance (< 1 mph) as implausible on the
low end too.

In [ ]:
speed_check = con.execute("""
    SELECT 
        COUNT(*) AS total_valid_duration_rows,
        SUM(CASE WHEN distance_miles / (EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp)) / 3600.0) > 100 
                 THEN 1 ELSE 0 END) AS speed_over_100mph,
        SUM(CASE WHEN distance_miles > 0 
                 AND distance_miles / (EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp)) / 3600.0) < 1 
                 THEN 1 ELSE 0 END) AS speed_under_1mph
    FROM taxi_all
    WHERE dropoff_timestamp > pickup_timestamp
""").fetchdf()

print(speed_check)

total_speed_anomalies = speed_check["speed_over_100mph"][0] + speed_check["speed_under_1mph"][0]
pct_speed_anomalies = 100 * total_speed_anomalies / total_rows
print(f"\nTotal speed anomalies: {total_speed_anomalies:,} ({pct_speed_anomalies:.6f}% of all rows)")

In [ ]:
# Combine everything into one master anomaly summary for the report
speed_rows = pd.DataFrame([
    {"anomaly": "speed_over_100mph", "count": int(speed_check["speed_over_100mph"][0]), 
     "pct_of_total": round(100*speed_check["speed_over_100mph"][0]/total_rows, 6)},
    {"anomaly": "speed_under_1mph", "count": int(speed_check["speed_under_1mph"][0]), 
     "pct_of_total": round(100*speed_check["speed_under_1mph"][0]/total_rows, 6)},
])

full_anomaly_df = pd.concat([anomaly_summary_df, speed_rows], ignore_index=True)
full_anomaly_df.to_csv("../data/interim/anomaly_summary_full.csv", index=False)
full_anomaly_df

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>anomaly</th>
      <th>count</th>
      <th>pct_of_total</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>negative_base_fare</td>
      <td>2400031</td>
      <td>4.938154</td>
    </tr>
    <tr>
      <th>1</th>
      <td>negative_charge_total</td>
      <td>875399</td>
      <td>1.801166</td>
    </tr>
    <tr>
      <th>2</th>
      <td>zero_distance_nonzero_fare</td>
      <td>1267110</td>
      <td>2.607127</td>
    </tr>
    <tr>
      <th>3</th>
      <td>zero_passengers</td>
      <td>231578</td>
      <td>0.476480</td>
    </tr>
    <tr>
      <th>4</th>
      <td>dropoff_before_pickup</td>
      <td>1942</td>
      <td>0.003996</td>
    </tr>
    <tr>
      <th>5</th>
      <td>zero_duration</td>
      <td>649668</td>
      <td>1.336716</td>
    </tr>
    <tr>
      <th>6</th>
      <td>speed_over_100mph</td>
      <td>11899</td>
      <td>0.024483</td>
    </tr>
    <tr>
      <th>7</th>
      <td>speed_under_1mph</td>
      <td>299516</td>
      <td>0.616265</td>
    </tr>
  </tbody>
</table>
</div>

### Investigating negative_base_fare (4.94% of all rows)

Hypothesis: negative fares correspond to voided/disputed trips (fare_settlement_method
codes 4 = Dispute, 6 = Voided trip per the data dictionary), not random corruption.

In [ ]:
negative_fare_breakdown = con.execute("""
    SELECT 
        fare_settlement_method,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_negative_fares
    FROM taxi_all
    WHERE base_fare < 0
    GROUP BY fare_settlement_method
    ORDER BY n DESC
""").fetchdf()

negative_fare_breakdown

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>fare_settlement_method</th>
      <th>n</th>
      <th>pct_of_negative_fares</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>0</td>
      <td>1694004</td>
      <td>70.58</td>
    </tr>
    <tr>
      <th>1</th>
      <td>4</td>
      <td>455956</td>
      <td>19.00</td>
    </tr>
    <tr>
      <th>2</th>
      <td>2</td>
      <td>162267</td>
      <td>6.76</td>
    </tr>
    <tr>
      <th>3</th>
      <td>3</td>
      <td>86005</td>
      <td>3.58</td>
    </tr>
    <tr>
      <th>4</th>
      <td>1</td>
      <td>1799</td>
      <td>0.07</td>
    </tr>
  </tbody>
</table>
</div>

In [ ]:
overlap_check = con.execute("""
    SELECT 
        SUM(CASE WHEN base_fare < 0 AND charge_total < 0 THEN 1 ELSE 0 END) AS neg_fare_and_neg_charge,
        SUM(CASE WHEN base_fare < 0 AND distance_miles = 0 THEN 1 ELSE 0 END) AS neg_fare_and_zero_dist,
        SUM(CASE WHEN dropoff_timestamp = pickup_timestamp AND base_fare < 0 THEN 1 ELSE 0 END) AS zero_dur_and_neg_fare,
        SUM(CASE WHEN dropoff_timestamp = pickup_timestamp AND distance_miles = 0 THEN 1 ELSE 0 END) AS zero_dur_and_zero_dist
    FROM taxi_all
""").fetchdf()

overlap_check

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>neg_fare_and_neg_charge</th>
      <th>neg_fare_and_zero_dist</th>
      <th>zero_dur_and_neg_fare</th>
      <th>zero_dur_and_zero_dist</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>869827.0</td>
      <td>204636.0</td>
      <td>98.0</td>
      <td>17043.0</td>
    </tr>
  </tbody>
</table>
</div>

### 5b. Negative Fare Investigation — Findings

**Hypothesis tested:** negative fares correspond to voided/disputed trips
(fare_settlement_method 4/6).

**Result:** Partially disproven.
| fare_settlement_method | % of negative fares |
|---|---|
| 0 (Flex Fare) | 70.58% |
| 4 (Dispute) | 19.00% |
| 2 (Cash) | 6.76% |
| 3 (No charge) | 3.58% |
| 1 (Credit card) | 0.07% |
| 6 (Voided) | 0% |

Negative fares are concentrated (95%+) in just two settlement methods (Flex Fare,
Dispute), suggesting a systemic pattern rather than random corruption — but not a
single clean cause at the time of this check. Voided trips (code 6), counter to
the initial hypothesis, show no negative fares at all.

> **Update (see Section 5.e):** The zero_distance_nonzero_fare investigation
> confirmed that Flex Fare (fare_settlement_method=0) trips are a structurally
> distinct record type — they also systematically lack `rate_class_id` and
> `distance_miles`. This explains the 70.58% Flex Fare share above: it is not
> a loose correlation but the same root cause (Flex Fare uses a different,
> undocumented data/pricing structure) driving both anomalies. The Dispute
> (19%) share remains a separate, smaller contributing factor, plausibly
> genuine fare reversals/adjustments.

**Decision (updated):** Negative `base_fare`/`charge_total` rows will be
**excluded from predictive modeling datasets** (fare prediction, ETA prediction).
Within this group, Flex Fare rows should be handled consistently with the
Flex Fare treatment decided in Section 5.e (retained as a documented segment,
excluded from standard fare/distance modeling) rather than treated as generic
corrupted data. Dispute-related negative fares (19%) are retained as a smaller,
separately-flagged subset. Given the concentration in two explainable
settlement types, this is now a **characterized business-logic segment**
rather than an ambiguous anomaly — still excluded from training data, but
with a clearer justification than "unexplained negative values."

### 5c. Anomaly Overlap Findings

- `negative_charge_total` is almost entirely a subset of `negative_base_fare`
  (869,827 / 875,399 = 99.4% overlap) — **treat as one combined anomaly**,
  not two independent ones, when reporting total affected rows.
- 204,636 rows have both negative fare AND zero distance — **now explained**:
  this is largely the Flex Fare population overlapping across both checks
  (see 5.e), not two coincidentally correlated issues.
- Zero-duration and negative-fare overlap only in 98 rows — largely
  independent anomaly populations.
- 17,043 / 649,668 (2.6%) zero-duration rows are also zero-distance;
  the remaining 97.4% have nonzero distance in zero recorded time —
  itself a speed-anomaly signature, likely already captured by the
  speed_over_100mph check.

### 5d. Investigating zero_distance_nonzero_fare (2.607% of all rows)

Hypothesis: these are not necessarily errors — could be negotiated fares,
group rides, or flat-rate trips (airport/Newark) where distance_miles wasn't
properly metered but a legitimate fare was still charged.

In [ ]:
zero_dist_by_rate_class = con.execute("""
    SELECT 
        rate_class_id,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_zero_dist_group
    FROM taxi_all
    WHERE distance_miles = 0 AND base_fare > 0
    GROUP BY rate_class_id
    ORDER BY n DESC
""").fetchdf()

zero_dist_by_rate_class

In [ ]:
# Also check fare_settlement_method breakdown for this group
zero_dist_by_settlement = con.execute("""
    SELECT 
        fare_settlement_method,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_zero_dist_group
    FROM taxi_all
    WHERE distance_miles = 0 AND base_fare > 0
    GROUP BY fare_settlement_method
    ORDER BY n DESC
""").fetchdf()

zero_dist_by_settlement

In [ ]:
# Check the actual fare magnitude distribution - are these small flat fees or full-size fares?
zero_dist_fare_stats = con.execute("""
    SELECT 
        MIN(base_fare) AS min_fare,
        MAX(base_fare) AS max_fare,
        AVG(base_fare) AS avg_fare,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY base_fare) AS median_fare,
        AVG(charge_total) AS avg_charge_total
    FROM taxi_all
    WHERE distance_miles = 0 AND base_fare > 0
""").fetchdf()

zero_dist_fare_stats

In [ ]:
# Check whether origin == destination for these rows (same-zone pickup/dropoff
# would support "trip barely moved" rather than "meter malfunction")
zero_dist_same_zone = con.execute("""
    SELECT 
        SUM(CASE WHEN origin_loc_id = dest_loc_id THEN 1 ELSE 0 END) AS same_zone,
        SUM(CASE WHEN origin_loc_id != dest_loc_id THEN 1 ELSE 0 END) AS diff_zone,
        COUNT(*) AS total
    FROM taxi_all
    WHERE distance_miles = 0 AND base_fare > 0
""").fetchdf()

zero_dist_same_zone

In [ ]:
flex_fare_check = con.execute("""
    SELECT 
        SUM(CASE WHEN fare_settlement_method = 0 AND rate_class_id IS NULL THEN 1 ELSE 0 END) AS both_conditions,
        SUM(CASE WHEN fare_settlement_method = 0 THEN 1 ELSE 0 END) AS flex_fare_total,
        SUM(CASE WHEN rate_class_id IS NULL THEN 1 ELSE 0 END) AS null_rate_total
    FROM taxi_all
    WHERE distance_miles = 0 AND base_fare > 0
""").fetchdf()

flex_fare_check

### 5e. Zero-Distance/Nonzero-Fare Findings — Root Cause Identified

**Discovery:** `rate_class_id IS NULL` (66.58% of this anomaly group, n=843,636)
exactly matches `fare_settlement_method = 0` (Flex Fare) row count. This is the
same population, and it overlaps substantially with the negative_base_fare
finding (Flex Fare = 70.58% of negative fares, section 5b).

**Conclusion:** Flex Fare trips are a structurally distinct fare/settlement type
that does not populate `rate_class_id` or `distance_miles` the same way standard
metered trips do, and shows an unusual (possibly non-standard sign convention)
fare pattern. This is not random data corruption — it is a systematic pattern
tied to one settlement method, likely reflecting a genuinely different pricing
mechanism (e.g., pre-negotiated app-based pricing) not detailed in the data
dictionary.

**Evidence against "trivial/junk records":**
- Median fare: $20.31, mean: $27.78 (comparable to normal trip fares, not
  near-zero cancellation fees)
- 72.1% have origin_loc_id != dest_loc_id (real point-to-point trips)

**Remaining minority explained by legitimate flat-rate types:**
- rate_class_id = 5 (Negotiated fare): 12.24%
- rate_class_id = 2/3/4 (JFK/Newark/Nassau-Westchester): 2.9%
- rate_class_id = 1 (Standard rate) with zero distance: 17.54% — likely genuine
  meter/GPS capture failures, since standard-rate trips should have real
  metered distance.
- rate_class_id = 99 (explicit Null/unknown): 0.71%

**Decision:**
1. Flex Fare (settlement_method=0) rows: retain as a separate documented
   segment. Exclude from `distance_miles`-dependent feature engineering and
   from the fare/ETA prediction training sets (their pricing mechanism
   doesn't follow the standard metered model), but do not delete — they are
   valid records of a real, different trip type and may warrant their own
   note in the business report (bonus track 6 candidate: "Flex Fare usage
   patterns").
2. Negotiated/airport rate_class rows (5, 2, 3, 4): keep, exclude only from
   distance-based feature calculations where distance=0 would be misleading.
3. Standard rate (rate_class_id=1) with zero distance (17.54%, ~222K rows):
   treat as genuine data capture errors — drop from modeling datasets, or
   impute distance using duration + average speed for the relevant zone pair
   if row volume warrants it.
4. rate_class_id NULL cases not explained by Flex Fare and the 99/unknown
   code represent a secondary data quality issue (missing rate class
   distinct from the documented null code) — worth a one-line note in the
   report as a labeling inconsistency.

### 5f. Investigating zero_passengers (0.476% of all rows, n=231,578)

Hypothesis: zero passenger counts are a data-entry/sensor gap rather than a
meaningful trip category, since taxis can't legitimately operate with zero
riders and collect a fare. Checking for correlation with provider, fare
type, and whether the rest of the trip data looks otherwise normal.

In [ ]:
zero_pax_by_provider = con.execute("""
    SELECT 
        provider_code,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_zero_pax_group
    FROM taxi_all
    WHERE rider_count = 0
    GROUP BY provider_code
    ORDER BY n DESC
""").fetchdf()

zero_pax_by_provider

In [ ]:
zero_pax_by_settlement = con.execute("""
    SELECT 
        fare_settlement_method,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_zero_pax_group
    FROM taxi_all
    WHERE rider_count = 0
    GROUP BY fare_settlement_method
    ORDER BY n DESC
""").fetchdf()

zero_pax_by_settlement

In [ ]:
# Check whether the rest of the trip looks "normal" - real distance, real fare, real duration
zero_pax_trip_stats = con.execute("""
    SELECT 
        AVG(distance_miles) AS avg_distance,
        AVG(base_fare) AS avg_fare,
        AVG(EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp))/60.0) AS avg_duration_min,
        SUM(CASE WHEN distance_miles = 0 THEN 1 ELSE 0 END) AS also_zero_distance,
        SUM(CASE WHEN base_fare < 0 THEN 1 ELSE 0 END) AS also_negative_fare,
        COUNT(*) AS total
    FROM taxi_all
    WHERE rider_count = 0
""").fetchdf()

zero_pax_trip_stats

In [ ]:
# Compare against overall dataset averages as a baseline
overall_stats = con.execute("""
    SELECT 
        AVG(distance_miles) AS avg_distance,
        AVG(base_fare) AS avg_fare,
        AVG(EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp))/60.0) AS avg_duration_min
    FROM taxi_all
    WHERE rider_count > 0
""").fetchdf()

overall_stats

### 5f. Zero Passengers — Findings

**Result:** Provider-specific sensor/reporting gap, not random corruption.

- 93.09% of zero-passenger rows come from provider_code=1 (Creative Mobile
  Technologies) — strongly suggests a vendor-specific onboard reporting issue
  rather than a distributed data entry error.
- Settlement methods are dominated by normal payment types (credit card 82.76%,
  cash 13.95%) — this population is distinct from the Flex Fare/negative-fare
  segment characterized in Sections 5b/5e.
- Trip characteristics (avg distance 2.55mi, avg fare $17.14, avg duration
  14.5min) are broadly consistent with the overall dataset baseline (3.58mi,
  $19.33, 17.5min) — these are real, complete trips, just missing the
  passenger count field.
- Overlap with other anomaly categories is minimal (3.8% also zero-distance,
  0.15% also negative-fare) — an independent, isolated issue.

**Decision:** Impute `rider_count = 0` with **1** (the standard/modal taxi
occupancy) rather than dropping these rows. Rationale: the rest of the trip
record is valid and usable for fare/ETA modeling; dropping ~231,578 rows
(0.476% of the dataset) would discard legitimate training data to fix a
field that has minimal predictive importance for fare/duration in the first
place. This will be flagged as an imputed-value flag column
(`rider_count_imputed = True`) rather than silently overwritten, to preserve
transparency for model interpretation and the report.

### 5g. Investigating zero_duration (1.337% of all rows, n=649,668)

Hypothesis: zero-duration rows (dropoff_timestamp = pickup_timestamp) are either
(a) meter glitches recording instant dropoff, or (b) part of the same Flex Fare /
negative-fare population already characterized in 5b/5e, given only 98 rows
overlapped with negative_fare directly but distance/settlement patterns may
still align.

In [ ]:
zero_dur_by_settlement = con.execute("""
    SELECT 
        fare_settlement_method,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_zero_dur_group
    FROM taxi_all
    WHERE dropoff_timestamp = pickup_timestamp
    GROUP BY fare_settlement_method
    ORDER BY n DESC
""").fetchdf()

zero_dur_by_settlement

In [ ]:
zero_dur_by_rate_class = con.execute("""
    SELECT 
        rate_class_id,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_zero_dur_group
    FROM taxi_all
    WHERE dropoff_timestamp = pickup_timestamp
    GROUP BY rate_class_id
    ORDER BY n DESC
""").fetchdf()

zero_dur_by_rate_class

In [ ]:
# Check distance and fare distribution for zero-duration rows
zero_dur_stats = con.execute("""
    SELECT 
        COUNT(*) AS total,
        SUM(CASE WHEN distance_miles = 0 THEN 1 ELSE 0 END) AS also_zero_distance,
        SUM(CASE WHEN distance_miles > 0 THEN 1 ELSE 0 END) AS nonzero_distance,
        AVG(distance_miles) AS avg_distance,
        MAX(distance_miles) AS max_distance,
        AVG(base_fare) AS avg_fare,
        SUM(CASE WHEN base_fare > 0 THEN 1 ELSE 0 END) AS positive_fare_count,
        SUM(CASE WHEN base_fare = 0 THEN 1 ELSE 0 END) AS zero_fare_count,
        SUM(CASE WHEN base_fare < 0 THEN 1 ELSE 0 END) AS negative_fare_count
    FROM taxi_all
    WHERE dropoff_timestamp = pickup_timestamp
""").fetchdf()

zero_dur_stats

In [ ]:
# Provider breakdown - check if this is also a vendor-specific issue like zero_passengers
zero_dur_by_provider = con.execute("""
    SELECT 
        provider_code,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_zero_dur_group
    FROM taxi_all
    WHERE dropoff_timestamp = pickup_timestamp
    GROUP BY provider_code
    ORDER BY n DESC
""").fetchdf()

zero_dur_by_provider

### 5g. Zero Duration — Findings

**Result:** Distinct, provider-specific timestamp-logging failure — not related
to the Flex Fare/negative-fare population (5b/5e) or zero_passengers (5f).

- 98.90% of zero-duration rows come from provider_code=7 (Helix) — a strong,
  near-exclusive concentration pointing to a vendor-specific bug logging
  identical pickup/dropoff timestamps.
- Settlement method (85.56% credit card, 12.95% cash) and rate_class (96.62%
  standard rate) are both normal — confirms this is independent from the
  Flex Fare / negative-fare pattern.
- 97.4% of zero-duration rows (632,625 / 649,668) have nonzero distance_miles
  — a physically impossible combination (distance traveled in zero elapsed
  time). Max recorded distance in this group is 22,046 miles, itself an
  extreme outlier confirming corrupted sensor data, not just a timestamp
  logging quirk.
- Minimal overlap with fare anomalies (98 negative, 450 zero fares out of
  649,668) — fare fields are largely unaffected; this is a distance/duration
  sensor issue specific to Helix's onboard system.

**Decision:** Drop all zero-duration rows (1.337% of total dataset). Unlike
zero_passengers, the core trip record here is internally contradictory
(nonzero distance with zero elapsed time) for the vast majority of cases,
making imputation unreliable — there's no trustworthy duration value to
infer when the paired distance field is itself often extreme/corrupted.
This should be documented as a known Helix (provider_code=7) data quality
issue in the report, worth flagging as a vendor integration recommendation
in the business findings.

### 5h. Investigating speed_under_1mph (0.616% of all rows, n=299,516)

Hypothesis: most of these are genuine short trips stuck in heavy traffic
(plausible, not an error) rather than a data quality issue. Need to check
whether duration is unreasonably long (suggesting a stuck meter or corrupted
timestamp) versus just slow-but-plausible city driving.

Note: excludes zero-duration rows already handled in 5g (avg speed calc
requires duration > 0).

In [ ]:
slow_speed_stats = con.execute("""
    SELECT 
        COUNT(*) AS total,
        AVG(distance_miles) AS avg_distance,
        MAX(distance_miles) AS max_distance,
        AVG(EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp))/60.0) AS avg_duration_min,
        MAX(EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp))/60.0) AS max_duration_min,
        AVG(base_fare) AS avg_fare
    FROM taxi_all
    WHERE dropoff_timestamp > pickup_timestamp
      AND distance_miles > 0
      AND distance_miles / (EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp)) / 3600.0) < 1
""").fetchdf()

slow_speed_stats

In [ ]:
# Break into duration buckets to see if this is "long duration, short distance"
# (plausible traffic) vs "extremely long duration" (likely stuck/corrupted trip)
slow_speed_duration_buckets = con.execute("""
    SELECT 
        CASE 
            WHEN dur_min <= 30 THEN '0-30 min'
            WHEN dur_min <= 60 THEN '30-60 min'
            WHEN dur_min <= 120 THEN '60-120 min'
            WHEN dur_min <= 360 THEN '2-6 hours'
            ELSE '6+ hours'
        END AS duration_bucket,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct,
        AVG(distance_miles) AS avg_distance
    FROM (
        SELECT distance_miles,
               EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp))/60.0 AS dur_min
        FROM taxi_all
        WHERE dropoff_timestamp > pickup_timestamp
          AND distance_miles > 0
          AND distance_miles / (EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp)) / 3600.0) < 1
    )
    GROUP BY duration_bucket
    ORDER BY n DESC
""").fetchdf()

slow_speed_duration_buckets

In [ ]:
# Provider check - consistent with the pattern of provider-specific issues seen so far
slow_speed_by_provider = con.execute("""
    SELECT 
        provider_code,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_group
    FROM taxi_all
    WHERE dropoff_timestamp > pickup_timestamp
      AND distance_miles > 0
      AND distance_miles / (EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp)) / 3600.0) < 1
    GROUP BY provider_code
    ORDER BY n DESC
""").fetchdf()

slow_speed_by_provider

### 5h. Speed Under 1mph — Findings

**Result:** Split population — majority genuine slow-traffic trips, minority
broken/stuck-meter records concentrated in one provider.

- max_duration_min = 14,880 min (~10.3 days) — confirms a genuine outlier tail,
  not merely "bad traffic."
- 93.5% of rows fall in the 0–60 min duration range with very short average
  distances (0.04–0.11 mi) — consistent with real gridlock/congestion, not
  an error. These should be **kept**.
- The remaining 6.5% (60+ min duration, n=19,349) show increasing distance
  alongside duration (up to 3.30 mi avg in the 6+ hour bucket) but even this
  is implausible for a real trip — indicates a stuck meter, unclosed app
  session, or corrupted timestamp rather than traffic. These should be
  **dropped**.
- 96.09% of this entire group comes from provider_code=2 (Curb Mobility) —
  a distinct vendor-specific issue from the zero-duration finding (5g, which
  was 98.90% Helix/provider_code=7). Two separate provider reliability
  issues identified, worth naming individually in the report.

**Decision:** Apply a duration threshold rather than a blanket rule — retain
speed_under_1mph rows with trip_duration <= 60 minutes (plausible traffic
delay), drop rows with trip_duration > 60 minutes (implausible stuck/broken
records). This affects ~19,349 rows (0.04% of total dataset) being dropped,
with the remaining ~280,167 rows retained as valid slow-traffic trips.

### 5i. General Outlier Pass — Extreme Distance Values

Surfaced during 5g: max_distance = 22,046.45 miles among zero-duration rows.
This check looks at extreme distances across the *entire* dataset, independent
of the duration anomaly, to see how widespread this is and what a sensible
upper bound looks like.

In [43]:
distance_percentiles = con.execute("""
    SELECT 
        MAX(distance_miles) AS max_distance,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY distance_miles) AS p99,
        PERCENTILE_CONT(0.999) WITHIN GROUP (ORDER BY distance_miles) AS p999,
        PERCENTILE_CONT(0.9999) WITHIN GROUP (ORDER BY distance_miles) AS p9999,
        AVG(distance_miles) AS avg_distance
    FROM taxi_all
""").fetchdf()

distance_percentiles

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,max_distance,p99,p999,p9999,avg_distance
0,397994.37,19.6,30.3,64.7,6.854183


In [44]:
# NYC + surrounding area (incl. rate classes like Nassau/Westchester) realistically
# caps single trips well under 100 miles. Check counts at a few candidate thresholds.
for threshold in [50, 100, 200, 500, 1000]:
    n = con.execute(f"SELECT COUNT(*) FROM taxi_all WHERE distance_miles > {threshold}").fetchone()[0]
    pct = 100 * n / total_rows
    print(f"> {threshold} miles: {n:,} rows ({pct:.6f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

> 50 miles: 9,444 rows (0.019431%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

> 100 miles: 2,817 rows (0.005796%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

> 200 miles: 2,125 rows (0.004372%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

> 500 miles: 1,949 rows (0.004010%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

> 1000 miles: 1,926 rows (0.003963%)


In [45]:
# Check provider concentration for extreme distances (>100mi), consistent
# with the provider-specific pattern seen in other anomalies
extreme_dist_by_provider = con.execute("""
    SELECT 
        provider_code,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_group
    FROM taxi_all
    WHERE distance_miles > 100
    GROUP BY provider_code
    ORDER BY n DESC
""").fetchdf()

extreme_dist_by_provider

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,provider_code,n,pct_of_group
0,2,2292,81.36
1,1,340,12.07
2,7,185,6.57


In [46]:
# Sanity check: for rows >100mi, does duration/fare scale plausibly with distance,
# or does the whole record look broken (like the zero-duration cases)?
extreme_dist_check = con.execute("""
    SELECT 
        distance_miles,
        EXTRACT(EPOCH FROM (dropoff_timestamp - pickup_timestamp))/60.0 AS duration_min,
        base_fare,
        provider_code,
        rate_class_id
    FROM taxi_all
    WHERE distance_miles > 100
    ORDER BY distance_miles DESC
    LIMIT 20
""").fetchdf()

extreme_dist_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,distance_miles,duration_min,base_fare,provider_code,rate_class_id
0,397994.37,18.000000,21.97,2,NaN
1,386088.43,20.000000,17.16,2,NaN
2,335783.69,11.000000,15.18,2,NaN
3,328522.20,22.000000,31.11,2,NaN
4,322576.17,7.000000,6.34,2,NaN
5,321242.41,46.000000,12.45,2,NaN
6,321242.41,45.833333,-12.23,2,NaN
7,320206.74,14.000000,23.18,2,NaN
8,318608.57,13.000000,20.03,2,NaN
9,317011.01,11.000000,-1.50,2,NaN


### 5i. Extreme Distance Values — Findings

**Result:** Confirmed sensor/GPS malfunction, not real trips — concentrated
in the same provider as the excessive-duration issue (5h).

- Massive gap between p99.99 (64.7 mi) and max (397,994 mi) — a small,
  clean outlier tail, not a heavy-tailed real distribution.
- Rows show identical, absurd distance values (e.g., 321,242.41 mi appearing
  twice with different fares/durations) — a fixed garbage sensor reading,
  not genuine measurement.
- Duration (single-to-double-digit minutes) and fare (single-to-double-digit
  dollars) remain completely normal for these rows — only distance_miles
  is corrupted.
- All affected rows have rate_class_id = NaN — same missing-metadata
  signature seen with the Flex Fare population (5e), though driven by a
  different provider here.
- 81.36% from provider_code=2 (Curb Mobility) — same provider responsible
  for the excessive-duration anomaly (5h). Reinforces Curb Mobility as
  having a broader sensor/GPS reliability issue, not just a meter/timer bug.

**Decision:** Drop rows with distance_miles > 100 (2,817 rows, 0.0058% of
total). Threshold chosen just above p99.99 (64.7 mi) with margin for
legitimate long trips (e.g., NYC to Nassau/Westchester), while excluding
the clearly corrupted tail. Duration/fare fields for these rows are not
independently corrupted, but the record as a whole is unusable for
distance-dependent modeling once distance_miles is discarded as garbage.

## Decisions Going Into Cleaning

1. **Schema/dtype consistency**: ✅ All 12 files share identical columns and dtypes — no harmonization needed.

2. **File integrity / date alignment**: ✅ All files correctly labeled, normal minor spillover at month boundaries. No mislabeled files.

3. **Corrupted timestamps (2008–2009 dates)**: 8 rows (~0%). **Drop.**

4. **Negative fares** (`base_fare`/`charge_total` < 0, 4.94%/1.80%, ~99% overlapping): Concentrated in Flex Fare (70.6%) and Dispute (19%) settlement methods — a systemic pattern, not random corruption. **Exclude from predictive modeling datasets**; retain as a documented flagged segment rather than delete.

5. **Zero distance, nonzero fare** (2.61%): 66.6% is the same Flex Fare population as #4 (missing `rate_class_id` too — a structurally distinct, undocumented fare type). Remainder mostly legitimate flat-rate trips (Negotiated/airport fares, ~15%) plus genuine standard-rate errors (17.5%). **Keep Flex Fare/flat-rate rows but exclude from distance-dependent features; drop standard-rate zero-distance rows (~222K) as capture errors.**

6. **Zero passengers** (0.48%): 93% from one provider (CMT) — a sensor/reporting gap, not corrupted trips (fare/distance/duration all normal). **Impute rider_count = 1**, flag with `rider_count_imputed`.

7. **Dropoff before pickup** (0.004%): Negligible. **Drop.**

8. **Zero duration** (1.34%): 98.9% from one provider (Helix); 97% also have nonzero (often extreme, up to 22,046 mi) distance — physically impossible, not just missing data. **Drop.**

9. **Speed > 100mph** (0.02%): Negligible. **Drop.**

10. **Speed < 1mph** (0.62%): 93.5% are short-distance/short-duration (≤60 min) — genuine slow traffic, **keep**. Remaining 6.5% (duration > 60 min, 96% from Curb Mobility) are implausible stuck-meter records. **Drop** (~19K rows).

11. **Extreme distance values** (>100 mi, 0.0058%): p99.99 = 64.7 mi vs. max = 397,994 mi — a small, clean outlier tail (GPS/sensor malfunction, not real distribution). Duration/fare remain normal for these rows; only `distance_miles` is corrupted, with duplicate absurd values (e.g., 321,242.41 mi appearing twice) confirming fixed garbage readings. 81.4% from Curb Mobility. **Drop rows with distance_miles > 100** (~2,817 rows).

**Provider-level pattern:** four distinct vendor-specific data quality issues identified:
- **CMT (provider 1):** missing passenger count
- **Helix (provider 7):** zero-duration / timestamp bug
- **Curb Mobility (provider 2):** stuck-meter / excessive duration **and** extreme/corrupted GPS distance readings

Worth a dedicated callout in the technical report — Curb Mobility shows the broadest reliability issue, spanning both time and distance sensors.

**Investigation phase complete.** All anomaly types from the original checklist (negative fares, zero distance, zero passengers, dropoff-before-pickup, unrealistic speed) plus the extreme-distance outlier pass are now characterized and decided. Ready to implement in the cleaning notebook.